In [ ]:
import pandas as pd
from sqlalchemy import create_engine
import datetime as dt
import time as tm
import numpy as np
from pymongo import MongoClient
from math import radians, cos, sin, asin, sqrt
import pymysql
from datetime import datetime,timedelta
import json
import urllib.request
import psycopg2
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore")




In [ ]:
%%HTML
<style type="text/css">
table.dataframe td, table.dataframe th {
    border: 1px  black solid !important;
  color:  black !important;
}

In [ ]:
import pandas as pd

from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 100000)



In [ ]:
import os
from config import *  # all DB / API handles come from environment variables, see config.py


In [ ]:
engine

In [ ]:
no_info=pd.read_sql("select * from fleet_lake.devicehistory where action='BATTERY_DISCONNECTED' and fromtime>{} limit 1000".format(tm.time()-3*86400),con)

In [ ]:
no_info['actual_from_time']=no_info['fromtime']-3600

In [ ]:
no_info['actual_to_time']=no_info['fromtime']

In [ ]:
vehicle_epochs=no_info[['vehicleid','actual_from_time','actual_to_time']].values.tolist()

In [ ]:
def loc_pings(l):
    try:
        
        vid=l[0]
        day_start=l[1]
        day_end=l[2]

        url=GPS_API_BASE+"/gps/vehicles?vehicleIds={}&fromTime={}&toTime={}&sortOrder=&timeCondition=".format(int(vid),int(day_start),int(day_end))
        data=json.load(urllib.request.urlopen(url))
        df=pd.DataFrame(data['data'][str(vid)])
        print(vid)
    except Exception as e:
        print(" loc pings issue >> " + str(e))
        df=pd.DataFrame()
    return df


In [ ]:
import multiprocessing
pool=multiprocessing.Pool(20)
results=pool.map(loc_pings,vehicle_epochs)
pool.close()
pool.join()



In [ ]:
yo=pd.concat(results)

In [ ]:
yo=yo[yo['speed']<5]

In [ ]:
pings=yo

In [ ]:
import sklearn
import hdbscan


In [ ]:
a=list(pings['location'])
points = np.array(a)
rads = np.radians(points)
clusterer = hdbscan.HDBSCAN(algorithm='best',alpha=1.0,
                            metric='haversine').fit(points)
cluster_labels = clusterer.fit_predict(points)
pings['cluster']=clusterer.labels_
pings['probabilities']=clusterer.probabilities_
pings=pd.concat([pings,pings['location'].apply(pd.Series).rename(columns={0:'longitude',1:'latitude'})],axis=1)


In [ ]:
pings.cluster.value_counts()

In [ ]:
len(pings)

In [ ]:
pings[pings['cluster']==551]['location'].value_counts()

In [ ]:
import folium
from folium import plugins
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from folium import FeatureGroup, LayerControl, Map, Marker


In [ ]:
import colorsys

def get_N_HexCol(N):
    HSV_tuples = [(x * 1.0 / N, 0.5, 0.5) for x in range(N)]
    hex_out = []
    for rgb in HSV_tuples:
        rgb = map(lambda x: int(x * 255), colorsys.hsv_to_rgb(*rgb))
        hex_out.append('#%02x%02x%02x' % tuple(rgb))
    return hex_out
color_palette = get_N_HexCol(clusterer.labels_.max()+2)


In [ ]:
m = folium.Map([28, 78], zoom_start=4)
plot=pings
for i,row in plot[plot['cluster']==330].iterrows():
    if row['cluster']>=0:
        folium.CircleMarker([row['latitude'],row['longitude']],
                            radius=2,
                            #popup=str(row['location']),
                            color=color_palette[row['cluster']]
                            #fill_color="#3db7e4", # divvy color
                            ).add_to(m)
    
    
m.save('j.html') 

In [ ]:
m.save('j.html') 

In [ ]:
plot.location=plot.location.apply(list)

In [ ]:
plot[plot['cluster']==835].location.value_counts()

In [ ]:
list(reversed([1,2,3]))


In [ ]:
import geopandas as gpd

In [ ]:
city0=gpd.read_file("data/IND_adm0.shp")
city1=gpd.read_file("data/IND_adm1.shp")
city2=gpd.read_file("data/IND_adm2.shp")
city3=gpd.read_file("data/IND_adm3.shp")

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
india=pd.concat([city2,city3])

In [ ]:
city3.plot(cmap='jet',column='NAME_1',figsize=(10,10))

In [ ]:
type(city3['geometry'].iloc[0])

In [ ]:
import geopandas
point = geopandas.GeoDataFrame.from_file("data/IND_adm1.shp") 
poly  = geopandas.GeoDataFrame.from_file("data/IND_adm1.shp")
from geopandas.tools import sjoin
pointInPolys = sjoin(point, poly, how='left')
grouped = pointInPolys.groupby('index_right')
grouped